# Target-side reader adaptation: frozen Qwen3.8-Flash-Next PLE → frozen Qwen3.5-0.8B (Kaggle P100)

**Goal:** transfer the frozen n-gram PLE (`Qwen/Qwen3.8-Flash-Next-FP8`, `e_t in R^2560`) to the frozen text backbone of `Qwen/Qwen3.5-0.8B` (`h_t in R^1024`, 24 layers) by training **only** a small target-side reader. Backbone and PLE stay frozen.

**Corrected protocol (v2):** validation always addresses PLE via `ngram_indices(token_ids)`; per-address deterministic `RandomPLE` (160-d rows → 2560-d); bijective per-head `PermutedPLE`; threshold-crossing checkpoints; two immutable validation artifacts (fast 65,536 / full 524,288 tokens, SHA256, never rebuilt inside training); **FineWeb-Edu only** for the first experiment (35B parity); exact PLE scale or abort; real-PLE requires `/kaggle/input` mount (no 48.7 GiB download into `/kaggle/working`); zero-based layer-index convention matching the 35B run; float16-first on P100 with FP32 reductions.

## Kaggle setup (do this first)

1. **Settings → Accelerator → `GPU P100`** (16 GB, sm_60). Notebook tries float16 first (RMSNorm/gate reductions stay FP32), falls back to float32 on instability.
2. **Settings → Internet ON** + secret **`HF_TOKEN`** (read on both Qwen repos).
3. **Attach the pinned PLE Kaggle Dataset** (shards + `manifest.json` + scale) as `/kaggle/input/<ple-ds>/`. Real-PLE cells **abort** without it — they never download 48.7 GiB into `/kaggle/working`.
4. Run top-to-bottom. **Run only the smoke/validation pass first; the 500K placement sweep cells are gated OFF until all correctness checks pass.**

In [ ]:
# Reader scale: frozen Kaggle stack and private HF credential.
import os, shutil, sys
from pathlib import Path
secret_value_0 = os.environ.get('HF_TOKEN')
if not secret_value_0:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_value_0 = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
assert secret_value_0 and secret_value_0.startswith('hf_'), 'set HF_TOKEN or attach Kaggle HF_TOKEN secret'
print('python', sys.version.split()[0], 'working disk', shutil.disk_usage('/kaggle/working'))
print('input mounts', sorted(str(p) for p in Path('/kaggle/input').iterdir()))


In [ ]:
# Cell 1 — deps (P100-compatible torch BEFORE first torch import)
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
if _cap.startswith('6.'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torch==2.5.1+cu118'],check=True)
    print('pinned P100 torch (cu118, sm_60 kernels)')
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchvision==0.20.1+cu118'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchaudio==2.5.1+cu118'],check=True)
    print('pinned matching torchvision+torchaudio (qwen3_5 modeling pulls both via media utils)')
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
torch.zeros(1).cuda()  # fail fast if the build lacks sm_60 kernels
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

## Config: dims, FineWeb-Edu-only corpus, layer convention

- **Source:** `Qwen/Qwen3.8-Flash-Next-FP8` — 2560-d, ngram 3, 8 heads/ngram → 16 slots × 160-d, `rows_per_part=2_500_012`, 128 parts, FP8 E4M3 + exact row scale (no fallback).
- **Target:** `Qwen/Qwen3.5-0.8B` — hidden 1024, 24 layers, vocab 248320, eos 248044, text-only load.
- **Layer numbering (matches 35B):** `IDX` = zero-based Python decoder index (`model.model.layers[IDX]`, what `injection_layers` stores); `HUMAN = IDX+1`. The 35B placements `(2,)/(13,)/(2,13)` were zero-based IDX; here `(2,)/(8,)/(2,8)` with `8 ≈ 24/3`.
- **First experiment corpus = FineWeb-Edu only** (35B parity). Mixed Stack/UltraChat/Cosmopedia comes only after transfer is proven.
- **Validation:** two immutable artifacts, fast 65,536 toks (128×512, iteration speed) and full 524,288 toks (1024×512, decisions), each SHA256-pinned, built once, reused everywhere.

In [ ]:
# Cell 2 — single config block (edit here only)
from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-0.8B'
    MEM_DIM: int = 2560
    HIDDEN: int = 1024
    N_LAYERS: int = 24
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8   # 16 address slots per token
    ROW_DIM: int = 160         # per-slot row dim; 16*160=2560
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234           # PLE hash seed — never change
    EOS: int = 248044  # config <|endoftext|>: PLE-training terminator (NOT chat <|im_end|> 248046)
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536    # 128 x 512
    VAL_FULL_TOKENS: int = 524288   # 1024 x 512 (35B parity)
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'  # FIRST experiment: FineWeb-Edu ONLY
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120    # correctness pass only (10 x 512); sweep stays OFF
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)  # threshold-crossing (not exact multiples)
    PLACEMENTS: tuple = ((2,), (8,), (2, 8))  # zero-based IDX, 35B convention
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3   # 0.0 = identity test mode
    SWEEP_ENABLED: bool = False  # DONE: 500K sweep complete, winner IDX 2+8
    BRANCH_ENABLED: bool = False  # DONE: controls persisted to ninnix/qwen-ple-reader-checkpoints
    REAL1M_ENABLED: bool = False  # DONE: v23 COMPLETE, bundles in dataset v4
    REAL5M_ENABLED: bool = False  # STAGED: 1M->2M->3M->5M legs, early-stop on saturate/reverse
    STAGEA_ENABLED: bool = False  # eval lives in kaggle_qwen35_08b_ple_eval.ipynb, never here

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(1,1),(2,1),(1,4),(2,4)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=1024; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,1024); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==24 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').rglob('manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
api=HfApi(token=tok)
trev=api.model_info(C.TARGET_ID).sha; srev=api.model_info(C.SOURCE_ID).sha
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)


In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/ple-08b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    from huggingface_hub import HfApi
    import os
    api=HfApi(token=secret_value_0)
    drev=api.dataset_info(C.DATASET_ID).sha; trev2=api.model_info(C.TARGET_ID).sha
    need_full=VALDIR/'validation-full.json'; need_fast=VALDIR/'validation-fast.json'
    if need_full.exists() and need_fast.exists():
        return json.loads(need_fast.read_text()), json.loads(need_full.read_text())
    ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=drev)
    arr=array('I'); docs=0
    for row in ds:
        docs+=1; t=row.get('text') or ''
        if t.strip(): arr.extend(tok.encode(t, add_special_tokens=False)); arr.append(C.EOS)  # training terminator
        if len(arr)>=C.VAL_FULL_TOKENS: del arr[C.VAL_FULL_TOKENS:]; break
    assert len(arr)==C.VAL_FULL_TOKENS, len(arr)
    base={'dataset':C.DATASET_ID,'config':C.DATASET_CONFIG,'dataset_rev':drev,'target_rev':trev2,'docs':docs,'skip_docs':docs}
    fast_arr=array('I', arr[:C.VAL_FAST_TOKENS])
    mf=_write_val('fast', fast_arr, base); mF=_write_val('full', arr, base)
    return mf, mF

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m

class FineWebEduStream:
    '''FIRST experiment: FineWeb-Edu only (35B parity). No Stack/UltraChat/Cosmopedia yet.'''
    def __init__(self, tok, skip_docs, dataset_rev):
        '''dataset_rev MUST be the pinned rev from frozen validation metadata — never re-resolve latest.'''
        from datasets import load_dataset
        assert dataset_rev, 'pass dataset_rev from validation-full.json'
        self.ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=dataset_rev).skip(skip_docs)
        self.it=iter(self.ds)  # streaming datasets are not iterators themselves
        self.tok=tok
    def __iter__(self): return self
    def __next__(self):
        while True:
            t=(next(self.it) or {}).get('text') or ''
            if str(t).strip(): return str(t)

def freeze_stream_tokens(tokenizer, stream, max_tokens, seq=512):
    '''EXACT token prefix train_reader consumes: same encode, same C.EOS, same chunking.
    Shared by the trainer and the compact-cache builder so both see identical tokens.'''
    buf = []
    out = []
    seen = 0
    while seen < max_tokens:
        while len(buf) < seq:
            buf += tokenizer.encode(next(stream), add_special_tokens=False) + [C.EOS]
        out.append(buf[:seq])
        buf = buf[seq:]
        seen += seq
    return torch.tensor(out, dtype=torch.long)

print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Cell 9 — frozen Qwen3.5-0.8B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
from huggingface_hub import HfApi
trev=HfApi(token=tok).model_info(C.TARGET_ID).sha
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==1024 and tconf.num_hidden_layers==24
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16, torch.float32]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
# Cell 10 — correctness gates (must ALL pass before any sweep): identity, frozen-backward, controls, scale
import torch, torch.nn.functional as F

def get_inj(layers, branches=1, gamma=0.0):
    # backbone keeps ACTIVE_DTYPE (fp16 if stable); reader params ALWAYS FP32 (AdamW in FP32)
    return ReaderInjection(model,list(layers),C.MEM_DIM,C.HIDDEN,branches,gamma).to('cuda')

probe=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
# (a) gamma=0 identity (fp16 tol looser)
inj=get_inj((2,),1,0.0); inj.set_memory(torch.randn(1,64,C.MEM_DIM).to('cuda'))
with torch.inference_mode(): a=model(input_ids=probe,use_cache=False).logits
inj.set_memory(None)
with torch.inference_mode(): b=model(input_ids=probe,use_cache=False).logits
dmax=(a.float()-b.float()).abs().max().item(); inj.close()
print('identity max|dlogit|:',dmax); assert dmax<(2e-3 if ACTIVE_DTYPE==torch.float16 else 1e-4)
# (b) frozen-backward: synthetic memory from the SAME toy ids passed to the model
toy=torch.randint(0,C.VOCAB,(1,32)).cuda()
inj=get_inj((8,),1,1e-3); inj.set_memory(RandomPLE([0.0]*16,[0.0086]*16).lookup(addresses(toy.cpu())).to('cuda'))
model.zero_grad(set_to_none=True)
loss=F.cross_entropy(getattr(model(input_ids=toy,use_cache=False),'logits')[:,:-1].float().reshape(-1,C.VOCAB), toy[:,1:].reshape(-1))
loss.backward()
rg=sum(1 for _,p in inj.named_parameters() if p.grad is not None); bg=sum(1 for p in model.parameters() if p.grad is not None)
print(f'loss {loss.item():.4f} reader_grads {rg} backbone_grads {bg}'); assert rg and not bg; inj.close()
# (c) RandomPLE: 16 addresses per token (one per head) -> (1,1,2560); deterministic; 1-addr flip changes output
_base=torch.tensor([[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]]])
r1=RandomPLE([0.0]*16,[0.0086]*16).lookup(_base); r2=RandomPLE([0.0]*16,[0.0086]*16).lookup(_base)
assert r1.shape==(1,1,2560) and torch.equal(r1,r2), (r1.shape, 'non-deterministic')
_alt=_base.clone(); _alt[0,0,0]=999999
assert not torch.equal(r1, RandomPLE([0.0]*16,[0.0086]*16).lookup(_alt)), 'changing one address must change the output'
print('RandomPLE 16-head layout + determinism + sensitivity ok', tuple(r1.shape))
# (d) PermutedPLE bijectivity + head preservation (uses RandomPLE base: no mount needed)
base=RandomPLE([0.0]*16,[0.0086]*16); perm=PermutedPLE(base)
H=16; test=torch.stack([torch.tensor([OFFS[h], OFFS[h]+1, OFFS[h]+SIZES[h]-1]) for h in range(H)],-1).unsqueeze(0)
idx=test.clone()
f=idx.long(); O=perm.O; A=perm.A; B=perm.B; S=perm.S
p=O+((f-O)*A+B)%S
assert all(int(p[0,i,h])>=OFFS[h] and int(p[0,i,h])<OFFS[h]+SIZES[h] for h in range(H) for i in range(3)), 'head boundary violated'
flat=set(int(x) for x in p.reshape(-1).tolist())
assert len(flat)==3*H, 'collision across sampled edge addresses'
print('PermutedPLE head-preserving bijective sample ok')
# (e) real-PLE mount + exact scale (aborts cleanly when dataset not attached)
try:
    real=MountPLE(); v=real.lookup(ngram_indices(torch.tensor([[1,2,3,4]])))
    assert v.shape==(1,4,2560) and torch.isfinite(v.float()).all()
    print('real PLE row ok scale=',real.scale,'std~',v.float().std().item())
except RuntimeError as e:
    print('real-PLE gate (expected until dataset attached):',str(e)[:220])
print('ALL CPU-capable correctness gates PASSED (mount gate informational))')

In [ ]:
# Cell 11 — training + eval core (val PASSED IN, never built here; threshold-crossing ckpts)
import math, time, json
import torch, torch.nn.functional as F

# addresses() lives in Cell 3 (single canonical tokens->PLE path)
def eval_loss(inj, store, val, use_reader=True):
    tot=ct=0; gates=[]
    with torch.inference_mode():
        for blk in val:
            blk=blk.unsqueeze(0).cuda()
            # FIX: ngram_indices BEFORE lookup — never raw token ids
            inj.set_memory(store.lookup(addresses(blk.cpu())).to('cuda') if use_reader else None)
            lg=model(input_ids=blk,use_cache=False).logits
            tot+=F.cross_entropy(lg[:,:-1].float().reshape(-1,lg.shape[-1]), blk[:,1:].reshape(-1), reduction='sum').item()
            ct+=blk[:,1:].numel()
            if use_reader:
                for r in inj.readers.values(): gates.append(r.last_gate.float().cpu().reshape(-1))
    l=tot/ct; g=torch.cat(gates) if gates else None
    import math as _m
    return {'loss':l,'ppl':_m.exp(min(l,20)),'gate_mean':g.mean().item() if g is not None else 0,'gate_std':g.std(correction=0).item() if g is not None else 0}

def train_reader(layers, branches=1, max_tokens=None, store=None, val=None, tag='run', ckpts=None, val_full=None, baselines=None, full_at=None, resume=None, cont_peak=None, cont_warm=200):
    # full_at: thresholds evaluated on val_full (default {max(ckpts)} = legacy behavior).
    # resume: ckpt dict from a previous train_reader save (reader+optimizer+scheduler+seen).
    # Every ckpt saves optimizer/scheduler/rng state so any leg can resume or persist.
    # val schedule: fast val below the final threshold, full val at the final threshold.
    # baselines: {'fast':..,'full':..} computed once and shared by all placements (None=recompute, smoke path).
    import random
    from torch.optim import AdamW
    assert val is not None, 'pass a frozen artifact from load_validation(); training must never build validation'
    max_tokens=max_tokens or C.SMOKE_TOKENS; ckpts=sorted(ckpts or [c for c in C.CKPTS if c<=max_tokens] or [max_tokens])
    random.seed(C.SEED); torch.manual_seed(C.SEED)
    inj=ReaderInjection(model,list(layers),C.MEM_DIM,C.HIDDEN,branches,C.GAMMA_INIT).to('cuda')  # FP32 reader+AdamW
    tr=[p for _,p in inj.named_parameters() if p.requires_grad]
    assert tr and not any(p.requires_grad for p in model.parameters())
    init_state={n: p.detach().cpu().clone() for n, p in inj.named_parameters()}
    def _unorm(pred):
        _t=0.0
        for _n,_p in inj.named_parameters():
            if pred(_n): _t+=(_p.detach().float().cpu()-init_state[_n].float()).pow(2).sum().item()
        return math.sqrt(_t)
    n_steps=max(1,max_tokens//C.SEQ); warm=max(1,math.ceil(n_steps*C.WARMUP_FRAC))
    opt=AdamW(tr,lr=C.LR,weight_decay=C.WD)
    _seen0=0; _skip=0
    if resume is not None:
        inj.load_state_dict({k: v.to('cuda') for k, v in resume['reader'].items()})
        opt.load_state_dict(resume['optimizer'])
        for _st in opt.state.values():
            for _k2, _v2 in _st.items():
                if isinstance(_v2, torch.Tensor): _st[_k2]=_v2.to('cuda')
        _seen0=int(resume['seen']); _skip=_seen0//C.SEQ
    # Continuation-aware LR: fresh cosine decays to 0 at its horizon, so naive restore
    # of a 1M-horizon scheduler into a 5M-horizon run restarts near peak (2.8e-05 shock).
    # If cont_peak is set with resume, do NOT load the old scheduler; instead re-warm
    # 0->cont_peak over cont_warm steps from the resume point, then cosine to 0 at 5M.
    if resume is not None and cont_peak is not None:
        _S0=int(resume['seen'])//C.SEQ; _S1=-(-max_tokens//C.SEQ); _fac=float(cont_peak)/C.LR
        def _cont_lam(s, _S0=_S0, _S1=_S1, _fac=_fac, _w=int(cont_warm)):
            if s<_S0+_w: return _fac*max(0.0,(s-_S0+1))/max(1,_w)
            _t=min((s-_S0-_w)/max(1,_S1-_S0-_w),1.0)
            return _fac*0.5*(1+math.cos(math.pi*_t))
        sch=torch.optim.lr_scheduler.LambdaLR(opt, _cont_lam)
        sch.last_epoch=_skip  # align to global step; first resumed step uses loaded ~0 LR, then ramps
        print('continuation schedule: S0 %d S1 %d peak %g warm %d (old scheduler NOT restored)' % (_S0,_S1,float(cont_peak),int(cont_warm)), flush=True)
    else:
        sch=torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s<warm else 0.5*(1+math.cos(math.pi*min((s-warm)/max(1,n_steps-warm),1))))
        if resume is not None and 'scheduler' in resume: sch.load_state_dict(resume['scheduler'])
    if resume is not None and 'torch_rng' in resume:
        torch.set_rng_state(resume['torch_rng']); torch.cuda.set_rng_state_all(resume['cuda_rng'])
    if resume is not None and 'init_state' in resume:
        init_state={n:p.clone() for n,p in resume['init_state'].items()}
    if baselines is None:
        base=eval_loss(inj,store,val,False); print('frozen baseline:',{k:round(v,5) if isinstance(v,float) else v for k,v in base.items()})
    else:
        base=None  # cached per-threshold baselines shared across placements
    _vmeta=json.loads((VALDIR/'validation-full.json').read_text())  # frozen val metadata pins corpus+rev
    skip=_vmeta['skip_docs']; drev=_vmeta['dataset_rev']  # skip val docs AND reuse pinned rev: no leakage, no drift
    stream=FineWebEduStream(tokenizer, skip, drev)
    chunks=freeze_stream_tokens(tokenizer, stream, max_tokens)  # EXACT shared prefix
    full_set=set(full_at) if full_at else {max(ckpts)}
    pending=[c for c in ckpts if c>_seen0]; res=[]; iloss=0; icount=0; t0=time.perf_counter()
    if resume is not None and _skip: chunks=chunks[_skip:]
    for step, _ids in enumerate(chunks, start=_skip+1):
        ids_t=_ids.unsqueeze(0)
        inj.set_memory(store.lookup(addresses(ids_t)).to('cuda'))
        opt.zero_grad(set_to_none=True)
        lg=model(input_ids=ids_t.cuda(),use_cache=False).logits
        loss=F.cross_entropy(lg[:,:-1].float().reshape(-1,lg.shape[-1]), ids_t.cuda()[:,1:].reshape(-1))
        if not torch.isfinite(loss): raise RuntimeError('non-finite loss — fallback to FP32 and rerun')
        loss.backward()
        assert not any(p.grad is not None for p in model.parameters()), 'backbone grad leak'
        gn=torch.nn.utils.clip_grad_norm_(inj.parameters(),1.0).item(); opt.step(); sch.step()
        seen=step*C.SEQ; iloss+=loss.item()*(C.SEQ-1); icount+=C.SEQ-1
        if step % 500 == 0: print('reader20 progress', seen, flush=True)
        while pending and seen>=pending[0]:  # threshold-crossing, not exact multiples
            th=pending.pop(0)
            _vv=val_full if (val_full is not None and th in full_set) else val
            pre_eval_save_reader(inj,seen,th)
            v=eval_loss(inj,store,_vv,True)
            _base=(baselines['full' if _vv is val_full else 'fast'] if isinstance(baselines,dict) else base)
            gam={n:p.item() for n,p in inj.named_parameters() if n.endswith('gamma')}
            r={'tokens':seen,'threshold':th,'val_set':('full' if _vv is val_full else 'fast'),'train_loss':iloss/icount,'val_loss':v['loss'],'dval':v['loss']-_base['loss'],
               'ppl':v['ppl'],'lr':float(opt.param_groups[0]['lr']),'gamma':gam,'gate':(v['gate_mean'],v['gate_std']),'gnorm':gn,
               'peak_GiB':round(torch.cuda.max_memory_allocated()/1024**3,3),'tok_s':round(seen/(time.perf_counter()-t0),1),
               'w_k_update':_unorm(lambda n: '.keys.' in n and n.endswith('.weight')),'w_v_update':_unorm(lambda n: n.endswith('.value.weight')),'beta_update':_unorm(lambda n: n.endswith('beta')),'rss_MiB':round(rss_mb(),1)}
            res.append(r); print(json.dumps(r,indent=1))
            import copy as _copy
            _osd=_copy.deepcopy(opt.state_dict())  # state_dict aliases live opt tensors: never .cpu() them in place
            _ckpt={'reader':{k:v.cpu() for k,v in inj.state_dict().items()},'cfg':(tuple(layers),branches),
                    'optimizer':_osd,
                    'scheduler':sch.state_dict(),'seen':seen,'iloss':iloss,'icount':icount,
                    'init_state':{n:p.cpu() for n,p in init_state.items()},
                    'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
            for _st in _ckpt['optimizer']['state'].values():
                for _k2,_v2 in _st.items():
                    if isinstance(_v2,torch.Tensor): _st[_k2]=_v2.cpu()
            torch.save(_ckpt, f'/kaggle/working/reader-{tag}-{seen}.pt')
            iloss=0; icount=0
    inj.close(); return {'baseline':base,'baselines':baselines,'checkpoints':res}

def write_bundle(outdir, tag, ckpt, run_meta, metrics):
    '''Canonical persistent artifact: reader.safetensors + reader.json + run.json + metrics.json + resume.pt.'''
    from safetensors.torch import save_file
    d=Path(outdir)/tag; d.mkdir(parents=True, exist_ok=True)
    save_file({k:v.cpu() for k,v in ckpt['reader'].items()}, str(d/'reader.safetensors'))
    (d/'reader.json').write_text(json.dumps({'format':'qwen-ple-reader','version':1,
        'source_model':C.SOURCE_ID,'target_model':C.TARGET_ID,'memory_dim':C.MEM_DIM,'hidden_dim':C.HIDDEN,
        'injection_layers_IDX':list(ckpt['cfg'][0]),'injection_layers_HUMAN':[i+1 for i in ckpt['cfg'][0]],
        'branches':ckpt['cfg'][1],'addressing':'qwen4-exp-splitmix64-xor-v1',
        'memory_condition':run_meta.get('memory_condition','real')},indent=2))
    (d/'run.json').write_text(json.dumps(run_meta,indent=2))
    (d/'metrics.json').write_text(json.dumps(metrics,indent=2))
    torch.save(ckpt, str(d/'resume.pt'))
    print('bundle wrote', sorted(p.name for p in d.glob('*')), flush=True)
    return d

print('trainer ready (threshold ckpts, val-injected, FineWeb-Edu-only stream)')


In [ ]:
# Reuse and verify the exact frozen validation bytes.
import hashlib, shutil
_hits = list(Path('/kaggle/input').rglob('transfer-shas.json'))
assert len(_hits) == 1
_man = json.loads(_hits[0].read_text())
for _name in ('frozen-v1__tokens-fast.uint32le', 'frozen-v1__tokens-full.uint32le',
              'frozen-v1__validation-fast.json', 'frozen-v1__validation-full.json'):
    _src = _hits[0].parent / _name
    assert hashlib.sha256(_src.read_bytes()).hexdigest() == _man[_name]['sha256']
    _dst = VALDIR / _name.split('__', 1)[1]
    shutil.copyfile(_src, _dst)
    assert hashlib.sha256(_dst.read_bytes()).hexdigest() == _man[_name]['sha256']
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Cell 12e - COMPACT working set: exact sweep prefix + full validation, one sequential pass over /kaggle/input.
# /kaggle/input stays the immutable master. Cache lives in /kaggle/tmp (WORK fallback). One cache serves IDX 2/8/2+8.
try:
    _cstore=MountPLE()
except RuntimeError as _e:
    print('COMPACT SKIPPED (attach PLE datasets first):', str(_e)[:200]); _cstore=None
if _cstore is not None:
    import time, hashlib, json as _json
    from array import array as _array
    _t0=time.perf_counter()
    _mfF=json.loads((VALDIR/'validation-full.json').read_text())
    _stream=FineWebEduStream(tokenizer, _mfF['skip_docs'], _mfF['dataset_rev'])
    _train_toks=freeze_stream_tokens(tokenizer, _stream, 20000000)[15000064//512:]
    _val,_=load_validation('full')
    _all=torch.cat([_val, _train_toks])
    _addrs=ngram_indices(_all)
    _uniq=torch.unique(_addrs.reshape(-1))
    _totreq=int(_addrs.numel()); _nunq=len(_uniq)
    print('requested addresses: %d; unique: %d; dedup ratio %.2fx' % (_totreq, _nunq, _totreq/_nunq), flush=True)
    _cdir=Path('/kaggle/tmp/compact-20m-suffix') if Path('/kaggle/tmp').exists() else WORK/'compact-20m-suffix'
    _cdir.mkdir(parents=True, exist_ok=True)
    print('cache dir:', _cdir, flush=True)
    _af=(_cdir/'addrs.u32').open('wb'); _rf=(_cdir/'rows.u8').open('wb')
    _ha=hashlib.sha256(); _hr=hashlib.sha256(); _n=0; _prev=-1; _readb=0; _ts=time.perf_counter()
    _parts=torch.div(_uniq, C.ROWS_PER_PART, rounding_mode='floor'); _local=_uniq % C.ROWS_PER_PART
    _byfile={}
    for _p in torch.unique(_parts).tolist():
        _byfile.setdefault(str(_cstore.part_paths[_p]), []).append(_p)
    from safetensors import safe_open as _so
    for _fpath in sorted(_byfile):
        with _so(_fpath, framework='pt', device='cpu') as _fh:
            for _p in sorted(_byfile[_fpath]):
                _full=_fh.get_slice(MountPLE.tensor_name(_p))[:]  # one sequential full-part read
                _readb+=_full.numel()
                _pos=torch.nonzero(_parts==_p).flatten()
                _rows=_full.index_select(0, _local.index_select(0, _pos))
                _au=_uniq.index_select(0, _pos)
                assert int(_au[0])>_prev; _prev=int(_au[-1])
                _ab=_array('I', _au.tolist()).tobytes(); _rb=_rows.view(torch.uint8).contiguous().numpy().tobytes()
                _af.write(_ab); _rf.write(_rb); _ha.update(_ab); _hr.update(_rb); _n+=_pos.numel()
    _af.close(); _rf.close()
    _scant=_ts and (time.perf_counter()-_ts)
    _meta={'format':'qwen-ple-compact','version':1,'token_train':int(_train_toks.numel()),'token_val':int(_val.numel()),
           'requested_addresses':_totreq,'address_count':_n,'dedup_ratio':_totreq/_n,
           'row_dim':C.ROW_DIM,'scale':float(_cstore.scale),'ple_revision':_cstore.ple_revision,
           'dataset_rev':_mfF['dataset_rev'],'train_skip_docs':_mfF['skip_docs'],
           'target_rev':_mfF.get('target_rev'),'tokenizer':'native token ids (Cell 7 semantic proof)',
           'hash':{'ngram':C.NGRAM,'heads_per_ngram':C.HEADS_PER_NGRAM,'vocab_size':C.VOCAB,'vocab_base':C.VOCAB_BASE,'seed':C.SEED,'eos':C.EOS},
           'addrs_sha256':_ha.hexdigest(),'rows_sha256':_hr.hexdigest(),'pattern':'freeze_stream_tokens+ngram_indices'}
    (_cdir/'compact.json').write_text(_json.dumps(_meta,indent=2))
    _dt=time.perf_counter()-_t0; _sz=(_cdir/'rows.u8').stat().st_size+(_cdir/'addrs.u32').stat().st_size
    print('COMPACT: %d unique rows (dedup %.2fx), %.3f GiB, built in %.0fs, source read %.0f MiB/s' % (_n,_totreq/_n,_sz/1024**3,_dt,_readb/_scant/1024**2), flush=True)
    _cmp=CompactPLE(_cdir)
    _g=torch.Generator().manual_seed(0)
    _samp=_uniq[torch.randint(0,len(_uniq),(2048,),generator=_g)].reshape(128,16)
    _d=(_cmp.lookup(_samp)-_cstore.lookup(_samp)).abs().max().item()
    print('equivalence max|diff| on 2048 sampled addresses:', _d); assert _d==0.0
    print('COMPACT BUILD OK (one cache serves IDX 2, 8, 2+8: placement never changes addresses)')

# Release construction tensors; CompactPLE keeps only the mapped cache.
import gc
for _name in ('_train_toks', '_all', '_addrs', '_uniq', '_parts', '_local', '_rows', '_full'):
    globals().pop(_name, None)
gc.collect()


In [ ]:
# Continue the verified REAL R=1 IDX2+IDX8 reader from 15M to 20M.
import hashlib
import json
import os
from pathlib import Path

import torch
from safetensors.torch import load_file, save_file

OUT = Path('/kaggle/working/reader-scale-20m')
OUT.mkdir(parents=True, exist_ok=True)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def pre_eval_save_reader(inj, seen, threshold):
    path = OUT / ('real-r1-%d.reader.safetensors' % seen)
    tmp = path.with_suffix('.safetensors.tmp')
    state = {k:v.detach().cpu() for k,v in inj.state_dict().items()}
    save_file(state, str(tmp))
    with open(tmp, 'rb+') as f:
        f.flush()
        os.fsync(f.fileno())
    reopened = load_file(str(tmp), device='cpu')
    assert set(reopened) == set(state)
    assert all(torch.equal(reopened[k], v) for k,v in state.items())
    os.replace(tmp, path)
    sha = digest(path)
    assert all(torch.equal(load_file(str(path), device='cpu')[k], v) for k,v in state.items())
    print('PRE-EVAL reader milestone', threshold, seen, sha, flush=True)

hits = list(Path('/kaggle/input').rglob('reader-15m-shas.json'))
assert len(hits) == 1, hits
source = json.loads(hits[0].read_text())
assert source['source_kernel'] == 'ninnix/qwen-ple-real-r1-reader-scale-15m-p100/1'
resume_name = 'real-15m-r1.resume.pt'
weights_name = 'real-15m-r1.reader.safetensors'
for name in (resume_name, weights_name):
    path = hits[0].parent / name
    record = source['files'][name]
    assert path.stat().st_size == record['size'] and digest(path) == record['sha256']
assert source['files'][resume_name]['sha256'] == '7d9369cd7a1c0bdacd3604746138a09b8fd2e5d7e9ae8212507ee4976cd86229'
assert source['files'][weights_name]['sha256'] == 'e4a760163ec07568178ab48aa533235a9af878183caf120d4e29ce1c8ce4b9dc'
cur = torch.load(hits[0].parent / resume_name, map_location='cpu', weights_only=False)
assert cur['seen'] == 15000064 and cur['cfg'] == ((2, 8), 1)
assert len(cur['optimizer']['state']) == 8 and 'scheduler' in cur
frozen15 = load_file(str(hits[0].parent / weights_name), device='cpu')
assert set(frozen15) == set(cur['reader'])
assert all(torch.equal(frozen15[k], cur['reader'][k]) for k in frozen15)
print('15M reader resume SHA/reopen and published weights verified', flush=True)

val_fast, mf = load_validation('fast')
val_full, mF = load_validation('full')
assert mf['tokens_sha256'] == '4c702b4313fca2fa7fc5ffaf242912ab48a1564fecda52dd01982e2b756e4abd'
assert mF['tokens_sha256'] == '26ffe65b1f6457d2557dbacc98197d025932057e3c183e3cf1de8eefb7c2fe7c'
assert mF['dataset_rev'] == '87f09149ef4734204d70ed1d046ddc9ca3f2b8f9'
meta = json.loads((_cdir / 'compact.json').read_text())
assert meta['token_train'] == 5000192 and meta['token_val'] == 524288
assert meta['dataset_rev'] == mF['dataset_rev'] and meta['train_skip_docs'] == mF['skip_docs']
store = _cmp

probe = ReaderInjection(model, [2, 8], C.MEM_DIM, C.HIDDEN, 1, C.GAMMA_INIT).to('cuda')
base_fast = eval_loss(probe, None, val_fast, False)
base_full = eval_loss(probe, None, val_full, False)
probe.close()
assert abs(base_fast['loss'] - 2.895546885152619) < 1e-4
assert abs(base_full['loss'] - 2.905584982696578) < 1e-4
baselines = {'fast':base_fast, 'full':base_full}
print('frozen disabled baselines', base_fast['loss'], base_full['loss'], flush=True)

run = train_reader([2, 8], 1, 20000000, store, val_fast, 'real-scale-20m-r1',
                   ckpts=[20000000], val_full=val_full, baselines=baselines,
                   full_at={20000000}, resume=cur, cont_peak=1.5e-5, cont_warm=200)
fin = run['checkpoints'][-1]
assert fin['tokens'] == 20000256 and fin['threshold'] == 20000000 and fin['val_set'] == 'full'
raw = Path('/kaggle/working/reader-real-scale-20m-r1-20000256.pt')
raw_sha = digest(raw)
ck = torch.load(raw, map_location='cpu', weights_only=False)
assert ck['seen'] == 20000256 and ck['cfg'] == ((2, 8), 1)
assert len(ck['optimizer']['state']) == 8
pre = OUT / 'real-r1-20000256.reader.safetensors'
weights = load_file(str(pre), device='cpu')
assert set(weights) == set(ck['reader'])
assert all(torch.equal(weights[k], ck['reader'][k]) for k in weights)
print('full reader resume SHA/reopen', raw_sha, flush=True)

run_meta = {
    'tag':'real-20m-r1', 'memory_condition':'real', 'tokens':20000256,
    'threshold':20000000, 'val_set':'full', 'placement_idx':[2, 8], 'branches':1,
    'seed':C.SEED, 'lr':C.LR, 'wd':C.WD, 'warmup_frac':C.WARMUP_FRAC,
    'gamma_init':C.GAMMA_INIT, 'target_rev':mF['target_rev'],
    'ple_revision':meta['ple_revision'], 'dataset_rev':mF['dataset_rev'],
    'train_skip_docs':mF['skip_docs'], 'val_fast_sha256':mf['tokens_sha256'],
    'val_full_sha256':mF['tokens_sha256'],
    'lr_schedule':{'mode':'continuation', 'peak':1.5e-5, 'warm_steps':200,
                   'horizon_tokens':20000000, 'base_lr':C.LR},
    'resume_parent_sha256':source['files'][resume_name]['sha256'],
    'origin':'verified 15M optimizer continuation',
}
metrics = {k:fin.get(k) for k in ('train_loss', 'val_loss', 'dval', 'ppl', 'gamma', 'gate', 'tok_s', 'peak_GiB')}
art = Path('/kaggle/working/qwen35-08b-ple-artifacts')
art.mkdir(parents=True, exist_ok=True)
bundle = write_bundle(art, 'real-20m-r1', ck, run_meta, metrics)
bundle_sha = {p.name:digest(p) for p in bundle.iterdir() if p.is_file()}
assert all(torch.equal(load_file(str(bundle / 'reader.safetensors'), device='cpu')[k], v) for k,v in weights.items())
reopened = torch.load(bundle / 'resume.pt', map_location='cpu', weights_only=False)
assert reopened['seen'] == 20000256 and len(reopened['optimizer']['state']) == 8
assert all(torch.equal(reopened['reader'][k], v) for k,v in weights.items())
summary = {'tokens':20000256, 'threshold':20000000,
           'full_val_raw_reader':fin['val_loss'], 'dval':fin['dval'],
           'gamma':fin['gamma'], 'gate':fin['gate'],
           'pre_eval_reader_sha256':digest(pre), 'raw_resume_sha256':raw_sha,
           'bundle_sha256':bundle_sha,
           'resume_parent_sha256':source['files'][resume_name]['sha256'],
           'reader_source':source['source_kernel']}
(OUT / 'reader-20m-summary.json').write_text(json.dumps(summary, indent=2))
print('READER 20M COMPLETE', summary, flush=True)
import shutil
shutil.copyfile(_cdir / 'compact.json', OUT / 'compact-20m.json')
shutil.rmtree(_cdir)
print('released completed 20M compact cache after checkpoint verification', flush=True)


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


In [ ]:
# Reader scale uses the frozen R=1 IDX2+IDX8 continuation only.


## Readout + push

- **Best IDX layer(s):** lowest full-val `dval` at 500K, monotonic 100K→250K→500K, stable gamma/gate (mean ~0.5–0.9, std < 0.25, no NaN).
- **Best way:** ship R=4 only on clear gain over R=1 at fixed budget/corpus/seed; GO iff real beats random + permuted + disabled with a downstream probe gain.
- **Push:** `kaggle kernels push -p <dir>` (see `kernel-metadata.json`: id `ninnix/qwen35-08b-ple-target-reader-p100`, P100, internet on). Add your PLE `dataset_sources` entry once the pinned shards dataset exists — real-PLE cells abort until then by design.